In [6]:
import pandas as pd
import re

print("Adım 1 Başlıyor: Veri Yükleme ve Temizleme...")

# 1. Veriyi Yüklüyoruz (Senin yerel dosya yolun)
dosya_yolu = r'/kaggle/input/datasets/yucelay/korelaktal/All_Data_Balanced.csv' 
df = pd.read_csv(dosya_yolu)

# Etiketleri ve metinleri doğru sütunlardan alalım
metin_sutunu = 'Hasta_Yorumu'
etiket_sutunu = 'Label'

# 2. Akıllı Metin Temizleme Fonksiyonu
def metin_temizle(text):
    if not isinstance(text, str):
        return ""
    # Çift tırnakları ve gereksiz özel karakterleri temizle (noktalama hariç)
    text = text.replace('"', '')
    text = text.replace('\n', ' ')
    # Birden fazla boşluğu tek boşluğa indir
    text = re.sub(r'\s+', ' ', text)
    # Baştaki ve sondaki boşlukları kırp
    text = text.strip()
    return text

# 3. Temizleme işlemini uygula
df['Temiz_Metin'] = df[metin_sutunu].apply(metin_temizle)

# 4. Durum Özeti Çıktısı
print("\n--- Veri Seti Özeti ---")
print(f"Toplam Veri Sayısı: {len(df)}")
print(f"Riskli/Hasta (1) Sayısı: {len(df[df[etiket_sutunu] == 1])}")
print(f"Sağlıklı/Risksiz (0) Sayısı: {len(df[df[etiket_sutunu] == 0])}")

print("\n--- Örnek Temizlenmiş Veri ---")
print(f"Metin: {df['Temiz_Metin'].iloc[0]}")
print(f"Etiketi: {df[etiket_sutunu].iloc[0]}")
print("\n✅ ADIM 1 TAMAMLANDI!")

Adım 1 Başlıyor: Veri Yükleme ve Temizleme...

--- Veri Seti Özeti ---
Toplam Veri Sayısı: 184000
Riskli/Hasta (1) Sayısı: 92000
Sağlıklı/Risksiz (0) Sayısı: 92000

--- Örnek Temizlenmiş Veri ---
Metin: Doktor bey merhaba, 57 yaşında bir erkek olarak bazı bilgilerimi vermek istiyorum. Öncelikle ailemde kanser vakası görüldü. Ben de sigara içen ve alkol alan birisiyim. Kilom obez düzeyinde, diyet/beslenme riskim ise orta olarak ölçüldü. Hareketlilik derseniz, hareketsiz denilebilir. Kronik olarak şekerim yok, bunun yanında bağırsak (iltihabi) sorunum yok. Durumum nedir?
Etiketi: 0

✅ ADIM 1 TAMAMLANDI!


In [7]:
from sklearn.model_selection import train_test_split
from transformers import ElectraTokenizer
from datasets import Dataset

print("Adım 2 Başlıyor: Veri Parçalama ve Tokenizasyon...")

# 1. Veriyi Eğitim (%80) ve Test (%20) olarak ayırıyoruz.
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['Temiz_Metin'].tolist(), 
    df['Label'].tolist(), 
    test_size=0.2, 
    random_state=42, 
    stratify=df['Label'] # Veri dengesini Eğitim ve Test setlerine de eşit dağıtır
)

print(f"Eğitim Seti (Modelin öğreneceği veri): {len(train_texts)} satır")
print(f"Test Seti (Modeli sınav yapacağımız veri): {len(val_texts)} satır")

# 2. Sözlüğü (Tokenizer) İndiriyoruz / Çağırıyoruz
MODEL_NAME = "dbmdz/electra-base-turkish-cased-discriminator"
print("\nSözlük (Tokenizer) yükleniyor...")
tokenizer = ElectraTokenizer.from_pretrained(MODEL_NAME)

# 3. Verileri PyTorch / Hugging Face Dataset formatına çeviriyoruz
train_dataset = Dataset.from_dict({'text': train_texts, 'label': train_labels})
val_dataset = Dataset.from_dict({'text': val_texts, 'label': val_labels})

# 4. Tokenizasyon İşlemi (Metinleri 128 uzunluğunda sayısal dizilere çevirir)
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

print("\n⏳ Metinler tokenlara çevriliyor... (Bilgisayarının hızına göre 1-2 dk sürebilir)")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

print("\n✅ ADIM 2 TAMAMLANDI! Metinler sayılara dönüştü.")

Adım 2 Başlıyor: Veri Parçalama ve Tokenizasyon...
Eğitim Seti (Modelin öğreneceği veri): 147200 satır
Test Seti (Modeli sınav yapacağımız veri): 36800 satır

Sözlük (Tokenizer) yükleniyor...


tokenizer_config.json:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]


⏳ Metinler tokenlara çevriliyor... (Bilgisayarının hızına göre 1-2 dk sürebilir)


Map:   0%|          | 0/147200 [00:00<?, ? examples/s]

Map:   0%|          | 0/36800 [00:00<?, ? examples/s]


✅ ADIM 2 TAMAMLANDI! Metinler sayılara dönüştü.


In [8]:
from transformers import ElectraForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch

print("Adım 3 Başlıyor: Model Kurulumu ve Eğitim...\n")

# 1. Metrik Hesaplama Fonksiyonu
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# 2. ELECTRA Modelini İndiriyoruz (Sadece 2 Sınıflı Kafa Ekliyoruz)
print("Modelin Zihni (Ağırlıklar) İndiriliyor...")
model = ElectraForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=2,
    id2label={0: "SAGLIKLI", 1: "RISKLI"}, 
    label2id={"SAGLIKLI": 0, "RISKLI": 1}
)

# 3. YEREL BİLGİSAYAR İÇİN GÜVENLİ EĞİTİM ARGÜMANLARI
training_args = TrainingArguments(
    output_dir="./electra_gecici_kayitlar",
    learning_rate=2e-5,                  
    per_device_train_batch_size=16,      # VRAM şişmesin diye 16'ya düşürdük
    gradient_accumulation_steps=2,       # Kalite düşmesin diye 2 adımı birleştirdik (Yani yine 32 gücünde)
    per_device_eval_batch_size=16,
    num_train_epochs=3,                  # Veri %50-50 dengeli olduğu için 3 tur harika sonuç verir
    weight_decay=0.01,
    eval_strategy="epoch",               
    save_strategy="epoch",               
    load_best_model_at_end=True,         
    metric_for_best_model="f1",
    fp16=True,                           # NVIDIA ekran kartın varsa hızı 2'ye katlar
    save_total_limit=2,                  # Harddiskini doldurmaması için sadece son 2 yedeği tutar
    logging_steps=500,                   
    report_to="none"                     # Windows telemetri hatalarını engeller
)

# 4. Standart Trainer Kurulumu (Veri Dengeli Olduğu İçin Ceza Sistemine Gerek Yok)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics
)

# 5. EĞİTİMİ BAŞLAT
print("\n🚀 YEREL EĞİTİM BAŞLIYOR! Lütfen bilgisayarı uyku moduna almayın...\n")
trainer.train()

# 6. Kesin Sonuçlar
print("\n--- 🎯 KESİN TEST SONUÇLARI ---")
eval_results = trainer.evaluate()

for key, value in eval_results.items():
    if key in ['eval_accuracy', 'eval_precision', 'eval_recall', 'eval_f1']:
        print(f"{key.replace('eval_', '').capitalize()}: {value:.4f}")

# 7. Modeli Senin Bilgisayarına (Yerel) Kaydet
kayit_klasoru = "./Kanser_Riski_Final_Model"
print(f"\n💾 En iyi model bilgisayarına kaydediliyor: {kayit_klasoru}")
trainer.save_model(kayit_klasoru)
tokenizer.save_pretrained(kayit_klasoru)
print("✅ EĞİTİM BİTTİ VE KAYDEDİLDİ! Artık LLM/RAG sistemine geçebiliriz.")

Adım 3 Başlıyor: Model Kurulumu ve Eğitim...

Modelin Zihni (Ağırlıklar) İndiriliyor...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: dbmdz/electra-base-turkish-cased-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream 

...

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye


--- 🎯 KESİN TEST SONUÇLARI ---


Accuracy: 0.6958
F1: 0.5629
Precision: 1.0000
Recall: 0.3917

💾 En iyi model bilgisayarına kaydediliyor: ./Kanser_Riski_Final_Model


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ EĞİTİM BİTTİ VE KAYDEDİLDİ! Artık LLM/RAG sistemine geçebiliriz.


In [ ]:
-----------------------------------------------------------------------------------------

In [7]:
!pip install -q transformers datasets accelerate scikit-learn

import pandas as pd
import re
import os
import shutil
import torch
from sklearn.model_selection import train_test_split
from transformers import ElectraTokenizer, ElectraForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("✅ HÜCRE 1 TAMAM: Kütüphaneler yüklendi.")

✅ HÜCRE 1 TAMAM: Kütüphaneler yüklendi.


In [9]:
dosya_yolu = '/kaggle/input/datasets/yucelay/korelaktal/All_Data_Balanced.csv' 
df = pd.read_csv(dosya_yolu)

def metin_temizle(text):
    if not isinstance(text, str): return ""
    return re.sub(r'\s+', ' ', text.replace('"', '').replace('\n', ' ')).strip()

df['Temiz_Metin'] = df['Hasta_Yorumu'].apply(metin_temizle)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['Temiz_Metin'].tolist(), df['Label'].tolist(), test_size=0.2, random_state=42, stratify=df['Label']
)
print(f"✅ HÜCRE 2 TAMAM: Veri temizlendi. Eğitim boyutu: {len(train_texts)}")

✅ HÜCRE 2 TAMAM: Veri temizlendi. Eğitim boyutu: 147200


In [10]:
MODEL_NAME = "dbmdz/electra-base-turkish-cased-discriminator"
tokenizer = ElectraTokenizer.from_pretrained(MODEL_NAME)

train_dataset = Dataset.from_dict({'text': train_texts, 'label': train_labels})
val_dataset = Dataset.from_dict({'text': val_texts, 'label': val_labels})

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

print("⏳ Tokenizasyon yapılıyor, lütfen bekleyin...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
print("✅ HÜCRE 3 TAMAM: Kelimeler sayılara dönüştürüldü.")

tokenizer_config.json:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

⏳ Tokenizasyon yapılıyor, lütfen bekleyin...


Map:   0%|          | 0/147200 [00:00<?, ? examples/s]

Map:   0%|          | 0/36800 [00:00<?, ? examples/s]

✅ HÜCRE 3 TAMAM: Kelimeler sayılara dönüştürüldü.


In [11]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    return {'accuracy': accuracy_score(labels, preds), 'f1': f1, 'precision': precision, 'recall': recall}

model = ElectraForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label={0: "SAGLIKLI", 1: "RISKLI"}, label2id={"SAGLIKLI": 0, "RISKLI": 1}
)

training_args = TrainingArguments(
    output_dir="/kaggle/working/electra_v3",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True, # Kaggle'ın Çift T4 GPU'sunu coşturacak ayar
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model, args=training_args, train_dataset=tokenized_train, eval_dataset=tokenized_val, compute_metrics=compute_metrics
)
print("✅ HÜCRE 4 TAMAM: Eğitim motoru (Trainer) hazır.")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: dbmdz/electra-base-turkish-cased-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream 

✅ HÜCRE 4 TAMAM: Eğitim motoru (Trainer) hazır.


In [12]:
print("\n🔥 EĞİTİM BAŞLIYOR... 🔥")
trainer.train()

# 1. Modeli Output Klasörüne Kaydet
kayit_yolu = '/kaggle/working/Kanser_Riski_Modeli_Final'
trainer.save_model(kayit_yolu)
tokenizer.save_pretrained(kayit_yolu)

# 2. Modeli Tek Parça ZIP Haline Getir
print("\n📦 Model başarıyla eğitildi. ZIP dosyasına dönüştürülüyor...")
shutil.make_archive('/kaggle/working/Kanser_Riski_Modeli', 'zip', kayit_yolu)
print("🎉 HER ŞEY BİTTİ! 'Kanser_Riski_Modeli.zip' dosyası Output klasöründe hazır.")


🔥 EĞİTİM BAŞLIYOR... 🔥


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.917484,0.915892,0.695842,0.562893,1.000000,0.391685
2,0.917556,0.911912,0.695842,0.562893,1.000000,0.391685
3,0.915956,0.911312,0.695842,0.562893,1.000000,0.391685


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


📦 Model başarıyla eğitildi. ZIP dosyasına dönüştürülüyor...
🎉 HER ŞEY BİTTİ! 'Kanser_Riski_Modeli.zip' dosyası Output klasöründe hazır.


In [14]:
import os
import shutil
from transformers import ElectraTokenizer
from IPython.display import FileLink

print("🧹 Gereksiz dosyalar atılıyor, Saf Model hazırlanıyor...")

# 1. Tertemiz bir vitrin klasörü açıyoruz
temiz_klasor = '/kaggle/working/Saf_Model'
os.makedirs(temiz_klasor, exist_ok=True)

# 2. SADECE modelin beynini ve ayarlarını o karmaşık klasörden kopyalıyoruz
eski_klasor = '/kaggle/working/electra_v3/checkpoint-2300'
shutil.copy(os.path.join(eski_klasor, 'model.safetensors'), temiz_klasor)
shutil.copy(os.path.join(eski_klasor, 'config.json'), temiz_klasor)

# 3. Eksik olan Sözlüğü (Tokenizer) internetten çekip doğrudan bu temiz klasöre kaydediyoruz
print("📚 Sözlük (Tokenizer) modele entegre ediliyor...")
tokenizer = ElectraTokenizer.from_pretrained("dbmdz/electra-base-turkish-cased-discriminator")
tokenizer.save_pretrained(temiz_klasor)

# 4. Tertemiz klasörü ZIP'liyoruz
print("📦 RAG için optimize edilmiş hafif model ZIP'leniyor...")
shutil.make_archive('/kaggle/working/Kanser_Riski_Modeli_Temiz', 'zip', temiz_klasor)

print("✅ İŞLEM TAMAM! Sadece ihtiyacın olan dosyalar paketlendi.")
print("Aşağıdaki linke tıklayarak indirebilirsin:")
display(FileLink('Kanser_Riski_Modeli_Temiz.zip'))

🧹 Gereksiz dosyalar atılıyor, Saf Model hazırlanıyor...
📚 Sözlük (Tokenizer) modele entegre ediliyor...
📦 RAG için optimize edilmiş hafif model ZIP'leniyor...
✅ İŞLEM TAMAM! Sadece ihtiyacın olan dosyalar paketlendi.
Aşağıdaki linke tıklayarak indirebilirsin:


/kaggle/working/Kanser_Riski_Modeli_Temiz.zip